# Geographic-Scale Dataset Build

Place-based dataset for the scaling sweep: top-N species become target classes, the rest pool into `non_target` with AudioSet ambient and BirdNET no-bird clips. Downloads cached.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import pyrootutils

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

# Parameters

In [ ]:
from building.polychirp_training import scale_slug

PLACE_NAME = "Paris"
N_TARGETS = 10

# Species to drop before the top-N pick (too few XC recordings to fill a class).
# Excluded species fall through into non_target.
EXCLUDE_SPECIES: list[str] = [
    "Anthus spinoletta",
]

# Area discovery coordinates (recorded in dataset.json).
LAT, LON = 48.8566, 2.3522
RADIUS_KM = 50

# Top-N get the full quota, the rest a small one.
TARGET_CLIPS_PER_SPECIES = 5000
NON_TARGET_CLIPS_PER_SPECIES = 250

# Cap on each non-top species' contribution to non_target.
NON_TARGET_PER_SPECIES_CAP = 300

# Area discovery (matches 00_area_analysis.ipynb defaults)
MIN_AREA_RECORDINGS = 10
ANALYSIS_MAX_BIRDNET = 2000
ANALYSIS_MIN_CONF = 0.85

# AudioSet ambient noise (diversified across about 60 FOCUS_CLASSES)
AS_CLIPS_PER_CLASS = 250
AS_MAX_TOTAL_CLIPS = 15000

# Collection slug: scale_<lat>_<lon>_r<radius> (e.g. "scale_49_2_r50").
COLLECTION_NAME = scale_slug(LAT, LON, RADIUS_KM)
print(f"collection: {COLLECTION_NAME}")


# Discover area species and pick top-N as targets

Species sorted by evidence (XC presence plus BirdNET detections); top-N are targets, the rest go to `non_target`.

In [ ]:
from building.analysis import (
    area_audio_cache_dir,
    bbox_from_radius,
    birdnet_area_analysis,
    combined_species_table,
    fetch_area_recordings,
    recordings_to_df,
)

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
area_df = recordings_to_df(await fetch_area_recordings(bbox))
bn = await birdnet_area_analysis(
    area_df,
    max_recordings=ANALYSIS_MAX_BIRDNET,
    min_confidence=ANALYSIS_MIN_CONF,
    cache_dir=area_audio_cache_dir(LAT, LON, RADIUS_KM),
)
combined = combined_species_table(
    area_df, bn["detections"], min_recordings=MIN_AREA_RECORDINGS
)
all_species = combined["scientific_name"].tolist()

# Drop excluded species, take top-N; the rest pool into non_target.
excluded = [sp for sp in EXCLUDE_SPECIES if sp in all_species]
unknown = [sp for sp in EXCLUDE_SPECIES if sp not in all_species]
candidates = [sp for sp in all_species if sp not in excluded]
TARGET_SPECIES = candidates[:N_TARGETS]
OTHER_SPECIES = [sp for sp in all_species if sp not in TARGET_SPECIES]

print(f"Top-{N_TARGETS} target species:")
for sp in TARGET_SPECIES:
    print(f"  {sp}")
if excluded:
    print(f"\nExcluded from targets (pooled into non_target): {excluded}")
if unknown:
    print(f"\nWARNING: EXCLUDE_SPECIES not found in area species: {unknown}")
print(f"\n{len(OTHER_SPECIES)} other species pooled into non_target")


# Write `dataset.json` (pre-download manifest)

Written before any download so the dataset identity survives an interrupted run.

In [ ]:
from building.polychirp_training.dataset import DatasetInfo, write_dataset_info, Audio, Budgets, Place

info = DatasetInfo(
    collection=COLLECTION_NAME,
    place=Place(name=PLACE_NAME, lat=LAT, lon=LON, radius_km=RADIUS_KM),
    n_targets=N_TARGETS,
    target_species=TARGET_SPECIES,
    other_species=OTHER_SPECIES,
    audio=Audio(), 
    budgets=Budgets(
        target_clips_per_species=TARGET_CLIPS_PER_SPECIES,
        non_target_clips_per_species=NON_TARGET_CLIPS_PER_SPECIES,
        non_target_per_species_cap=NON_TARGET_PER_SPECIES_CAP,
        audioset_clips_per_class=AS_CLIPS_PER_CLASS,
        audioset_max_total=AS_MAX_TOTAL_CLIPS,
    ),
)
print(write_dataset_info(info))

# Stream-download target species (heavy budget)

In [ ]:
from building.data.download import download_and_process
from building.data.sources import XCSource, EBirdSource

with XCSource() as xc, EBirdSource() as eb:
    for sp in TARGET_SPECIES:
        await download_and_process(sp, TARGET_CLIPS_PER_SPECIES, [xc, eb])

# Stream-download other species (light budget, pooled as non-target)

In [ ]:
with XCSource() as xc, EBirdSource() as eb:
    for sp in OTHER_SPECIES:
        await download_and_process(sp, NON_TARGET_CLIPS_PER_SPECIES, [xc, eb])

# Boost `no_bird` via OTHER_SPECIES reprocessing

Recordings with no BirdNET detection above the threshold yield a mined no-bird clip. Doubling the OTHER_SPECIES quota harvests more; stops at `NO_BIRD_CAP` or when no recordings remain.

In [ ]:
from building.data.download import (
    BIRDNET_NO_BIRD_DIR, NO_BIRD_CAP, SUBSAMPLES_DIR, download_and_process,
)
from building.data.sources import XCSource, EBirdSource

NO_BIRD_TARGET = NO_BIRD_CAP   # 20_000


def n_no_bird() -> int:
    return sum(1 for _ in BIRDNET_NO_BIRD_DIR.glob("*.wav")) if BIRDNET_NO_BIRD_DIR.exists() else 0


def total_other_clips() -> int:
    return sum(
        sum(1 for _ in (SUBSAMPLES_DIR / sp.replace(" ", "_")).glob("*.wav"))
        for sp in OTHER_SPECIES
        if (SUBSAMPLES_DIR / sp.replace(" ", "_")).exists()
    )


curr_nb, curr_ot = n_no_bird(), total_other_clips()
print(f"start  no_bird={curr_nb}/{NO_BIRD_TARGET}  other_total={curr_ot}")

quota = NON_TARGET_CLIPS_PER_SPECIES
done = False
while not done and curr_nb < NO_BIRD_TARGET:
    quota *= 2
    print(f"\n>>> boost OTHER_SPECIES quota -> {quota}/source")
    with XCSource() as xc, EBirdSource() as eb:
        for sp in OTHER_SPECIES:
            before = n_no_bird()
            await download_and_process(sp, quota, [xc, eb], verbose=False)
            after = n_no_bird()
            print(f"  {sp:<32s} no_bird {before:>5d} -> {after:>5d} (+{after - before})")
            if after >= NO_BIRD_TARGET:
                done = True
                break
    new_nb, new_ot = n_no_bird(), total_other_clips()
    print(
        f"after pass: no_bird={new_nb} (+{new_nb - curr_nb})  "
        f"other_total={new_ot} (+{new_ot - curr_ot})"
    )
    if new_ot == curr_ot:
        print("no new OTHER_SPECIES clips - sources exhausted, stopping")
        break
    curr_nb, curr_ot = new_nb, new_ot

print(f"\nend    no_bird={curr_nb}/{NO_BIRD_TARGET}")

# Verify downloads

Check each species' clips on disk against its budget and the cached listings.

`flag`:
- ok: at budget.
- maxed: below budget but every listed recording is processed; the source has no more.
- REFETCH: below budget and recordings remain; re-run the download cells to top it up.

A source reading 0 clips means its key is missing from `building/.env`.

In [ ]:
import csv

import pandas as pd

from building.data.download import LISTINGS_DIR, SUBSAMPLES_DIR

SOURCES = [("xc", "XC"), ("ebird", "ML")]


def recs(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open(newline="") as f:
        return sum(1 for _ in csv.DictReader(f))


def audit(species: list[str], per_source: int) -> pd.DataFrame:
    target = per_source * len(SOURCES)
    rows: list[dict[str, object]] = []
    for sp in species:
        key = sp.replace(" ", "_")
        folder = SUBSAMPLES_DIR / key
        row: dict[str, object] = {"species": sp}
        total = pending = 0
        for name, prefix in SOURCES:
            avail = recs(LISTINGS_DIR / "available" / name / f"{key}.csv")
            done = recs(LISTINGS_DIR / "processed" / name / f"{key}.csv")
            clips = len(list(folder.glob(f"{prefix}*.wav"))) if folder.exists() else 0
            row |= {f"{name}_avail": avail, f"{name}_done": done, f"{name}_clips": clips}
            total += clips
            pending += max(0, avail - done)
        row["total"] = total
        row["target"] = target
        row["flag"] = "ok" if total >= target else ("REFETCH" if pending else "maxed")
        rows.append(row)
    return pd.DataFrame(rows)


targets = audit(TARGET_SPECIES, TARGET_CLIPS_PER_SPECIES)
others = audit(OTHER_SPECIES, NON_TARGET_CLIPS_PER_SPECIES)
display(targets)
display(others)

both = pd.concat([targets, others], ignore_index=True)
for name, _ in SOURCES:
    if both[f"{name}_clips"].sum() == 0:
        print(f"WARNING: source {name!r} produced 0 clips - check {name.upper()}_API_KEY in building/.env")
ok = int((both["flag"] == "ok").sum())
refetch = both.loc[both["flag"] == "REFETCH", "species"].tolist()
print(f"{ok}/{len(both)} species at target.")
if refetch:
    print(f"re-fetchable (recordings still available, re-run the download cells): {refetch}")

# Stream-download AudioSet ambient clips

In [ ]:
from building.data.audioset import AudioSetConfig, stream_download_audioset_async

as_cfg = AudioSetConfig(
    clips_per_class=AS_CLIPS_PER_CLASS,
    max_total_clips=AS_MAX_TOTAL_CLIPS,
)
results = await stream_download_audioset_async(as_cfg)

# Assemble train/val/test

Each top-N species is its own class; `non_target` balances audioset, xc_other and birdnet_no_bird, capped per non-top species at `NON_TARGET_PER_SPECIES_CAP`.

In [ ]:
from building.data.dataset import build_task_dataset

dataset_path = build_task_dataset(
    COLLECTION_NAME,
    target_species=TARGET_SPECIES,
    non_target_species=OTHER_SPECIES,
    non_target_per_species_cap=NON_TARGET_PER_SPECIES_CAP,
)
print(dataset_path)